# Model Serving: FastAPI, Docker, Kubernetes & Dynamic Micro-Batching

Serving machine learning models in production requires bridging raw model computation with scalable web services:
1. **FastAPI Application**: High-performance asynchronous REST endpoints with Pydantic type validation.
2. **Kubernetes Health Probes**: `/healthz` (liveness) and `/ready` (readiness) probes ensuring zero-downtime rolling updates.
3. **Dynamic Micro-Batching**: Grouping concurrent single requests into vectorized batch operations without violating SLAs.
4. **Containerization & Kubernetes Architecture**: Multi-stage Docker packaging, resource limits, and Horizontal Pod Autoscaling (HPA).

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('07-mlops/serving/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
import serving_engine
print('serving_engine loaded')

## 1. FastAPI Model Server with Health Probes
Production endpoints exposing single prediction, batch prediction, and Kubernetes liveness/readiness probes.

In [ ]:
from starlette.testclient import TestClient
from serving_engine import MockProductionModel, create_app

# Instantiate production model and mount in FastAPI
model = MockProductionModel(version="v1.4.0", weights=[0.8, -0.4, 1.2, 0.1])
app = create_app(model)
client = TestClient(app)

# 1. Kubernetes probes
resp_live = client.get("/healthz")
resp_ready = client.get("/ready")
print(f"Liveness  probe ({resp_live.status_code}): {resp_live.json()}")
print(f"Readiness probe ({resp_ready.status_code}): {resp_ready.json()}")

# 2. Real-time single inference request
payload = {"features": [1.5, 0.2, -0.8, 0.4]}
resp_pred = client.post("/predict", json=payload)
print(f"\nSingle prediction response: {resp_pred.json()}")

# 3. High-throughput batch inference request
batch_payload = {
    "items": [
        {"features": [1.0, 0.0, 0.5, 0.0]},
        {"features": [0.0, 2.0, -1.0, 0.3]},
        {"features": [-0.5, 0.5, 0.0, 1.0]}
    ]
}
resp_batch = client.post("/predict_batch", json=batch_payload)
print(f"\nBatch prediction response ({resp_batch.json()['batch_size']} items): {resp_batch.json()['predictions']}")

## 2. Dynamic Micro-Batching
Balancing single-request latency with hardware throughput on Tensor Cores.

In [ ]:
from serving_engine import DynamicMicroBatcher

batcher = DynamicMicroBatcher(model, max_batch_size=8, max_wait_ms=10.0)

# Simulate incoming requests
requests = [
    [0.1, 0.2, 0.3, 0.4],
    [0.9, -0.1, 0.5, 0.2],
    [-0.3, 0.8, 0.1, 0.0]
]

for i, req in enumerate(requests, 1):
    preds, count = batcher.add_request(req)
    print(f"Request {i} processed via micro-batch size {count} -> Pred: {preds[0]:.4f}")

## 3. Production Dockerfile & Kubernetes Deployment Manifest
Multi-stage Docker builds and declarative Kubernetes specifications.

In [ ]:
from serving_engine import generate_dockerfile, generate_k8s_deployment

print("--- Generated Multi-Stage Dockerfile ---")
print(generate_dockerfile()[:280] + "\n... [TRUNCATED] ...")

print("\n--- Generated Kubernetes Deployment & Service Manifest ---")
print(generate_k8s_deployment(app_name="recommendation-service", replicas=3)[:340] + "\n... [TRUNCATED] ...")